# 8) Tiny Mentor Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Give a student one encouraging line plus one micro-challenge based on their daily progress. Avoid repeating recent challenges.

## Simple rules

- Output is short.
- Avoid repeating last 3 challenges.
- Save history in memory.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "08_tiny_mentor_memory.json"

PRAISE = [
    "Consistency beats intensity.",
    "Small wins compound.",
    "Momentum is a force.",
    "You showed up. That matters.",
]
CHALLENGES = [
    "Write 2 test cases for your code.",
    "Rename 3 unclear variables.",
    "Explain your solution in 5 lines.",
    "Solve 1 extra easy problem.",
    "Refactor one function into smaller parts.",
]

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}
    history = memory.get("last_challenges", [])

    Tools.notify("Tiny Mentor Agent is running. Type 'stop' to end.")

    while True:
        progress = Tools.ask("What did you do today (1 line)?")
        if should_stop(progress):
            memory["last_challenges"] = history[-10:]
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Bye!")
            break

        praise = random.choice(PRAISE)
        options = [c for c in CHALLENGES if c not in history[-3:]] or CHALLENGES
        challenge = random.choice(options)
        history.append(challenge)

        Tools.notify(f"You said: {progress}\n\nPraise: {praise}\nMicro-challenge: {challenge}")
        memory["last_challenges"] = history[-10:]
        env.execute(Action("save_memory", {}), memory)

run_agent()


## Easy extensions

- Add modes: strict / friendly / QA.
- Add a streak counter.
- Let students add their own challenges.